# 03 Transformer Model

This notebook implements a Transformer-based sentiment classifier using a pretrained DistilBERT model.

The model uses:

- Hugging Face tokenization
- DistilBERT sequence classification
- Fine-tuning for SST-5 sentiment classification
- Accuracy and macro-F1 evaluation


## Download and Prepare the Dataset

Run the following code to download and load the data into pandas DataFrames.

Each sample has:

- text: a short movie review snippet
- label: an integer (0-4) representing sentiment
- label_text: a human-readable version of the label

In [ ]:
import gdown
import pandas as pd

# Download SST-5 JSONL files
train_url = "https://drive.google.com/uc?id=1ERmAINKSVZ-cvnTDvgajcOlIFqu2Swdk"
test_url = "https://drive.google.com/uc?id=1Vw-SyeYij8ZspQuAzIuRizNK_If4-6iP"

train_json = "sst5_train.jsonl"
test_json = "sst5_test.jsonl"

gdown.download(train_url, train_json, quiet=False)
gdown.download(test_url, test_json, quiet=False)

# Load into pandas DataFrames
train_df = pd.read_json(train_json, lines=True)
test_df = pd.read_json(test_json, lines=True)

# Preview the data
train_df.head()

Downloading...
From: https://drive.google.com/uc?id=1ERmAINKSVZ-cvnTDvgajcOlIFqu2Swdk
To: /content/sst5_train.jsonl
100%|██████████| 1.32M/1.32M [00:00<00:00, 107MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Vw-SyeYij8ZspQuAzIuRizNK_If4-6iP
To: /content/sst5_test.jsonl
100%|██████████| 343k/343k [00:00<00:00, 19.4MB/s]


,text,label,label_text
0,"a stirring , funny and finally transporting re...",4,very positive
1,apparently reassembled from the cutting-room f...,1,negative
2,they presume their audience wo n't sit still f...,1,negative
3,the entire movie is filled with deja vu moments .,2,neutral
4,this is a visually stunning rumination on love...,3,positive


## Your Task

Your goal is to train a model (e.g., using scikit-learn, PyTorch, or transformers) that takes text as input and predicts the label (integer from 0 to 4). You can use any model architecture and preprocessing strategy you prefer.

Your final model should generate predictions on the test set. The result must be a list of integers in the following format:

`preds = [1, 0, 2, 4, 3, ...]`  

Make sure your prediction list has **the same length and order as test_df**.

In [ ]:
# 1. Imports
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm

# 2. Basic settings
model_name = "distilbert-base-uncased"
max_length = 128
batch_size = 16
num_epochs = 2
learning_rate = 2e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# 3. Train / validation split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["text"].astype(str).tolist(),
    train_df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"]
)

test_texts = test_df["text"].astype(str).tolist()

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))
print("Test size:", len(test_texts))

# 4. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 5. Dataset class
class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

# Create datasets
train_dataset = TextDataset(train_texts, train_labels)
val_dataset = TextDataset(val_texts, val_labels)
test_dataset = TextDataset(test_texts)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 6. Load model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=learning_rate)

# 7. Validation function
def evaluate(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    acc = accuracy_score(all_labels, all_preds)

    return avg_loss, macro_f1, acc

# 8. Training loop
best_f1 = 0
best_model_state = None

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        progress_bar.set_postfix(train_loss=loss.item())

    avg_train_loss = total_train_loss / len(train_loader)
    val_loss, val_f1, val_acc = evaluate(model, val_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val Macro F1: {val_f1:.4f}")
    print(f"Val Accuracy: {val_acc:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_model_state = model.state_dict()

print("\nBest validation macro F1:", best_f1)

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)


# 9. Predict on test set
model.eval()
preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        batch_preds = torch.argmax(logits, dim=1).cpu().numpy().tolist()
        preds.extend(batch_preds)

print("Number of predictions:", len(preds))
print("Number of test samples:", len(test_df))
print("First 10 predictions:", preds[:10])

Using device: cpu
Train size: 6835
Validation size: 1709
Test size: 2210


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/2:   0%|          | 0/428 [00:00<?, ?it/s]


Epoch 1
Train Loss: 1.2676
Val Loss:   1.1621
Val Macro F1: 0.4841
Val Accuracy: 0.4857


Epoch 2/2:   0%|          | 0/428 [00:00<?, ?it/s]


Epoch 2
Train Loss: 0.9836
Val Loss:   1.1448
Val Macro F1: 0.4744
Val Accuracy: 0.5108

Best validation macro F1: 0.4840822303477652


Predicting:   0%|          | 0/139 [00:00<?, ?it/s]

Number of predictions: 2210
Number of test samples: 2210
First 10 predictions: [1, 0, 3, 2, 0, 1, 4, 4, 3, 4]


## Evaluation

After generating your predictions, run the following code to evaluate your model.

The classification report includes precision, recall, and f1-score. The support column shows the number of true instances for each class. In addition to per-class metrics, the report also provides overall performance through accuracy, macro average (unweighted mean across classes), and weighted average (average weighted by support).

In [ ]:
from sklearn.metrics import classification_report

# True labels from test set
true_labels = test_df['label'].tolist()

# Your predictions (replace this with your actual predictions)
# preds = [...]

# Evaluation report
print(classification_report(true_labels, preds, digits=4, target_names=[
    "very negative", "negative", "neutral", "positive", "very positive"
]))


               precision    recall  f1-score   support

very negative     0.6016    0.2652    0.3682       279
     negative     0.5461    0.7299    0.6247       633
      neutral     0.3896    0.2494    0.3041       389
     positive     0.4849    0.6627    0.5601       510
very positive     0.6678    0.4937    0.5677       399

     accuracy                         0.5285      2210
    macro avg     0.5380    0.4802    0.4850      2210
 weighted avg     0.5334    0.5285    0.5107      2210

